# VocEd Lab 03-v2 — Colour-Based Thresholding

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emilsar/VocEd/blob/main/03v2_stain_deconvolution.ipynb)

Lab 03 showed that Gaussian blur, non-local means denoising, and morphological opening/closing
produced only marginal improvements over Lab 02.  Why?  Because they all wrapped the same
grayscale threshold — and if nucleus and cytoplasm overlap in brightness, no amount of
pre-processing can fix that.

This lab attacks the problem at the *representation* level.  H&E stained slides encode cell
identity in **colour**: haematoxylin stains nuclei dark purple; eosin stains cytoplasm pink.
Converting to grayscale throws that information away entirely.

We test two colour-based representations, both requiring only arithmetic on the RGB channels:

- **Option A — B−R difference:** nuclei are purple (B > R), cytoplasm is pink (R > B).  A single
  channel-difference map isolates nuclei far better than brightness alone.
- **Option B — HSV saturation + hue:** saturation separates stained tissue from white background;
  hue separates purple nuclei from pink cytoplasm within the stained region.

**New pipeline (both options):**  
`RGB → colour feature → per-feature thresholds → morphology → Bayesian optimisation`

By the end of this lab you will be able to:
- Explain why grayscale conflates nucleus and cytoplasm in H&E images.
- Compute channel-difference and HSV representations and interpret them visually.
- Build and compare two colour threshold pipelines using Dice score.
- Identify which colour representation gives the most separable features for this dataset.
- Compute N/C ratio predictions and the Pearson correlation coefficient.

## 0. Setup

In [ ]:
!pip install scikit-optimize scikit-image --quiet

# Clone the repo (skip if already present)
!git clone https://github.com/emilsar/VocEd.git 2>/dev/null || echo 'Already cloned.'
%cd VocEd

## 1. Load Data & Recreate the Train/Test Split

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
import skimage.color as skc
import skimage.morphology as skm
from scipy.ndimage import binary_fill_holes
from skopt import gp_minimize
from skopt.space import Real

N = len(glob.glob('imagedata/X/*.npy'))
X = np.stack([np.load(f'imagedata/X/{i}.npy') for i in range(N)])
y = np.stack([np.load(f'imagedata/y/{i}.npy') for i in range(N)])

# Same split as every previous lab — same seed, same indices
np.random.seed(42)
idx       = np.random.permutation(N)
train_idx = idx[:160]
test_idx  = idx[160:]

def to_gray(img):
    """Standard ITU-R grayscale from a (3, H, W) float32 image."""
    return 0.299 * img[0] + 0.587 * img[1] + 0.114 * img[2]

def segment_gray(img, t_nuc, t_bg):
    """Lab 01/02 grayscale threshold pipeline — kept here for comparison."""
    gray = to_gray(img)
    pred = np.zeros(gray.shape, dtype=np.int64)
    pred[gray < t_nuc]                           = 2
    pred[(gray >= t_nuc) & (gray < t_bg)]        = 1
    return pred

def dice_score(pred, target, cls):
    p = pred   == cls
    t = target == cls
    inter = (p & t).sum()
    denom = p.sum() + t.sum()
    return 1.0 if denom == 0 else 2 * inter / denom

print(f'Loaded {N} images.  Train: {len(train_idx)}  Test: {len(test_idx)}')

## 2. Colour Feature Exploration

Grayscale reduces the three-dimensional colour signal to a single brightness value.
This collapses purple nuclei and pink cytoplasm onto the same scale, leaving no clean
threshold between them.

Two colour representations that preserve the distinction:

**B−R difference (Option A)**

Purple pixels (nuclei) absorb more green/yellow and transmit more blue, so $B > R$.
Pink pixels (cytoplasm) absorb more blue and transmit more red, so $R > B$.
White background has $B \approx R$.  The difference $B - R$ therefore gives:

$$B - R > 0 \Rightarrow \text{nucleus candidate} \qquad B - R < 0 \Rightarrow \text{cytoplasm candidate}$$

**HSV — Hue and Saturation (Option B)**

Converting RGB to Hue-Saturation-Value separates *what colour* (H) from *how bright* (V):

- Background: $S \approx 0$ (white = unsaturated, regardless of brightness)
- Nucleus: $S > 0$ and hue in the blue-violet range ($H \approx 0.65$–$0.82$)
- Cytoplasm: $S > 0$ and hue in the pink-red range ($H \approx 0.82$–$1.0$)

By thresholding on S first we cleanly remove background without touching V at all;
a second threshold on H then separates the two stained classes.

In [ ]:
IDX  = 7
img7 = X[IDX].transpose(1, 2, 0)     # (H, W, 3) for skimage

# Option A: B−R difference
br7  = img7[:, :, 2] - img7[:, :, 0]   # positive = more blue = nucleus

# Option B: HSV channels
hsv7 = skc.rgb2hsv(img7)
H7, S7, V7 = hsv7[:, :, 0], hsv7[:, :, 1], hsv7[:, :, 2]

gray7 = to_gray(X[IDX])

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes[0, 0].imshow(img7);                          axes[0, 0].set_title('Original RGB')
axes[0, 1].imshow(gray7,  cmap='gray');           axes[0, 1].set_title('Grayscale (Labs 01–03)')
axes[0, 2].imshow(br7,    cmap='RdBu');           axes[0, 2].set_title('B − R  (Option A)\nblue=nucleus, red=cytoplasm')
axes[1, 0].imshow(H7,     cmap='hsv',  vmin=0, vmax=1);  axes[1, 0].set_title('Hue  (Option B)')
axes[1, 1].imshow(S7,     cmap='gray', vmin=0, vmax=1);  axes[1, 1].set_title('Saturation  (Option B)')
axes[1, 2].imshow(V7,     cmap='gray', vmin=0, vmax=1);  axes[1, 2].set_title('Value  (Option B)')
for ax in axes.flat:
    ax.axis('off')
plt.tight_layout()
plt.show()

# Mean feature values per ground-truth class — how well does each feature separate the classes?
mask7 = y[IDX]
print('Mean feature values by ground-truth class:')
print(f'  Grayscale:  nucleus={gray7[mask7==2].mean():.3f}  cytoplasm={gray7[mask7==1].mean():.3f}  background={gray7[mask7==0].mean():.3f}')
print(f'  B−R:        nucleus={br7[mask7==2].mean():.3f}   cytoplasm={br7[mask7==1].mean():.3f}   background={br7[mask7==0].mean():.3f}')
print(f'  Hue H:      nucleus={H7[mask7==2].mean():.3f}   cytoplasm={H7[mask7==1].mean():.3f}   background={H7[mask7==0].mean():.3f}')
print(f'  Saturation: nucleus={S7[mask7==2].mean():.3f}   cytoplasm={S7[mask7==1].mean():.3f}   background={S7[mask7==0].mean():.3f}')

## 3. Option A — B−R Difference Pipeline

Three stages:

1. **B−R difference** — one subtraction; no model, no calibration.
2. **Hybrid thresholding:**
   - `B − R > t_nuc` → nucleus
   - `grayscale > t_bg` → background (background is white = high grayscale regardless of colour)
   - everything else → cytoplasm
3. **Morphology** — same targeted operations as Lab 03:
   `binary_fill_holes` → `remove_small_objects` → `closing`.

In [ ]:
def segment_rgb(img, t_nuc, t_bg, use_morph=True, morph_radius=2):
    """
    Option A: B−R difference pipeline.

    img          : (3, H, W) float32, channels-first
    t_nuc        : B−R threshold — pixels with B−R > t_nuc become nucleus (class 2)
    t_bg         : grayscale threshold — pixels brighter than t_bg become background (class 0)
    """
    br_diff = img[2] - img[0]          # B − R: positive → more blue → nucleus
    gray    = to_gray(img)

    nuc_mask  = br_diff > t_nuc
    bg_mask   = gray    > t_bg
    cyto_mask = ~nuc_mask & ~bg_mask   # cytoplasm: everything in between

    if use_morph:
        disk = skm.disk(morph_radius)
        nuc_mask  = binary_fill_holes(nuc_mask)
        nuc_mask  = skm.remove_small_objects(nuc_mask, min_size=50)
        nuc_mask  = skm.closing(nuc_mask, disk)
        cyto_mask = binary_fill_holes(cyto_mask)
        cyto_mask = skm.remove_small_objects(cyto_mask, min_size=50)
        cyto_mask = cyto_mask & ~nuc_mask

    pred = np.zeros(br_diff.shape, dtype=np.int64)
    pred[cyto_mask] = 1
    pred[nuc_mask]  = 2
    return pred


pred_rgb7 = segment_rgb(X[7], t_nuc=0.05, t_bg=0.85)
print('Classes present:', np.unique(pred_rgb7))

## 4. Option B — HSV Pipeline

Three stages:

1. **RGB → HSV conversion** — separates colour identity (H) from brightness (V).
2. **Two-step thresholding on S and H:**
   - `S < t_sat` → background (white = unsaturated, irrespective of brightness)
   - among stained pixels: `H < t_hue` → nucleus (purple/violet hue); else → cytoplasm (pink/red)
3. **Same morphological post-processing** as Option A.

The advantage over Option A: saturation is a more robust background detector than grayscale
brightness — a slightly darker background region is still white and still has $S \approx 0$.

In [ ]:
def segment_hsv(img, t_sat, t_hue, use_morph=True, morph_radius=2):
    """
    Option B: HSV saturation + hue pipeline.

    img          : (3, H, W) float32, channels-first
    t_sat        : saturation threshold — pixels with S < t_sat become background (class 0)
    t_hue        : hue threshold — among stained pixels, H < t_hue → nucleus (class 2);
                   H >= t_hue → cytoplasm (class 1)
    """
    img_hwc = img.transpose(1, 2, 0)
    hsv     = skc.rgb2hsv(img_hwc)        # returns H, S, V all in [0, 1]
    H       = hsv[:, :, 0]
    S       = hsv[:, :, 1]

    bg_mask   = S < t_sat                 # background: white = low saturation
    nuc_mask  = (~bg_mask) & (H < t_hue) # nucleus: stained + purple/violet hue
    cyto_mask = ~nuc_mask & ~bg_mask      # cytoplasm: stained + pink/red hue

    if use_morph:
        disk = skm.disk(morph_radius)
        nuc_mask  = binary_fill_holes(nuc_mask)
        nuc_mask  = skm.remove_small_objects(nuc_mask, min_size=50)
        nuc_mask  = skm.closing(nuc_mask, disk)
        cyto_mask = binary_fill_holes(cyto_mask)
        cyto_mask = skm.remove_small_objects(cyto_mask, min_size=50)
        cyto_mask = cyto_mask & ~nuc_mask

    pred = np.zeros(H.shape, dtype=np.int64)
    pred[cyto_mask] = 1
    pred[nuc_mask]  = 2
    return pred


pred_hsv7 = segment_hsv(X[7], t_sat=0.10, t_hue=0.83)
print('Classes present:', np.unique(pred_hsv7))

In [ ]:
## 5. Visual Comparison on Image 7

pred_gray7 = segment_gray(X[7], 0.45, 0.85)
pred_rgb7  = segment_rgb( X[7], t_nuc=0.05, t_bg=0.85)
pred_hsv7  = segment_hsv( X[7], t_sat=0.10, t_hue=0.83)

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
axes[0].imshow(X[7].transpose(1, 2, 0));                                   axes[0].set_title('Original RGB')
axes[1].imshow(y[7],        cmap='tab10', vmin=0, vmax=9, interpolation='nearest'); axes[1].set_title('Ground truth')
axes[2].imshow(pred_gray7,  cmap='tab10', vmin=0, vmax=9, interpolation='nearest'); axes[2].set_title('Grayscale (Lab 02)')
axes[3].imshow(pred_rgb7,   cmap='tab10', vmin=0, vmax=9, interpolation='nearest'); axes[3].set_title('Option A — B−R diff')
axes[4].imshow(pred_hsv7,   cmap='tab10', vmin=0, vmax=9, interpolation='nearest'); axes[4].set_title('Option B — HSV')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

# Dice scores with hand-picked starting thresholds (pre-optimisation)
print('Image 7 — hand-picked thresholds (pre-optimisation):')
for name, pred in [('grayscale', pred_gray7), ('B−R diff', pred_rgb7), ('HSV', pred_hsv7)]:
    d1 = dice_score(pred, y[7], 1)
    d2 = dice_score(pred, y[7], 2)
    print(f'  {name:12s}  cytoplasm={d1:.3f}  nucleus={d2:.3f}  mean={0.5*(d1+d2):.3f}')

## 6. Bayesian Optimisation

We run Bayesian optimisation separately for each new pipeline, plus re-run Lab 02 as the
grayscale baseline — all with the same budget (50 calls) and the same train/test split.

| Pipeline | Parameters | Search range |
|---|---|---|
| Grayscale (Lab 02) | `t_nuc`, `t_bg` | [0.10, 0.70], [0.50, 0.99] |
| Option A — B−R diff | `t_nuc` (B−R), `t_bg` (grayscale), `morph_radius` | [−0.10, 0.30], [0.50, 0.99], [1, 5] |
| Option B — HSV | `t_sat`, `t_hue`, `morph_radius` | [0.05, 0.40], [0.60, 0.95], [1, 5] |

> **Note:** three optimisations × 50 evaluations × 160 images each.
> Expect **8–15 minutes** depending on hardware.

In [ ]:
# ── Grayscale baseline (Lab 02) ───────────────────────────────────────────────
def objective_gray(params):
    t_nuc, t_bg = params
    scores = [( dice_score(segment_gray(X[i], t_nuc, t_bg), y[i], 1)
              + dice_score(segment_gray(X[i], t_nuc, t_bg), y[i], 2)) / 2
              for i in train_idx]
    return -np.mean(scores)

print('Optimising grayscale baseline…')
res_gray = gp_minimize(objective_gray,
                       [Real(0.10, 0.70), Real(0.50, 0.99)],
                       n_calls=50, n_initial_points=10, random_state=42, verbose=False)
best_gray = res_gray.x
print(f'  t_nuc={best_gray[0]:.4f}  t_bg={best_gray[1]:.4f}  train Dice={-res_gray.fun:.4f}')

# ── Option A: B−R difference ──────────────────────────────────────────────────
def objective_rgb(params):
    t_nuc, t_bg, radius = params
    scores = [(dice_score(segment_rgb(X[i], t_nuc, t_bg, morph_radius=int(radius)), y[i], 1)
             + dice_score(segment_rgb(X[i], t_nuc, t_bg, morph_radius=int(radius)), y[i], 2)) / 2
              for i in train_idx]
    return -np.mean(scores)

print('Optimising Option A (B−R diff)…')
res_rgb = gp_minimize(objective_rgb,
                      [Real(-0.10, 0.30), Real(0.50, 0.99), Real(1, 5)],
                      n_calls=50, n_initial_points=10, random_state=42, verbose=False)
best_rgb = res_rgb.x
print(f'  t_nuc={best_rgb[0]:.4f}  t_bg={best_rgb[1]:.4f}  radius={best_rgb[2]:.1f}  train Dice={-res_rgb.fun:.4f}')

# ── Option B: HSV ─────────────────────────────────────────────────────────────
def objective_hsv(params):
    t_sat, t_hue, radius = params
    scores = [(dice_score(segment_hsv(X[i], t_sat, t_hue, morph_radius=int(radius)), y[i], 1)
             + dice_score(segment_hsv(X[i], t_sat, t_hue, morph_radius=int(radius)), y[i], 2)) / 2
              for i in train_idx]
    return -np.mean(scores)

print('Optimising Option B (HSV)…')
res_hsv = gp_minimize(objective_hsv,
                      [Real(0.05, 0.40), Real(0.60, 0.95), Real(1, 5)],
                      n_calls=50, n_initial_points=10, random_state=42, verbose=False)
best_hsv = res_hsv.x
print(f'  t_sat={best_hsv[0]:.4f}  t_hue={best_hsv[1]:.4f}  radius={best_hsv[2]:.1f}  train Dice={-res_hsv.fun:.4f}')

print('\nAll optimisations complete.')

## 7. Test-Set Dice Comparison

In [ ]:
def test_dice_mean(pred_fn):
    scores = [(dice_score(pred_fn(i), y[i], 1) + dice_score(pred_fn(i), y[i], 2)) / 2
              for i in test_idx]
    return np.mean(scores)

d_lab01  = test_dice_mean(lambda i: segment_gray(X[i], 0.45, 0.85))
d_lab02  = test_dice_mean(lambda i: segment_gray(X[i], best_gray[0], best_gray[1]))
d_rgb    = test_dice_mean(lambda i: segment_rgb( X[i], best_rgb[0], best_rgb[1], morph_radius=int(best_rgb[2])))
d_hsv    = test_dice_mean(lambda i: segment_hsv( X[i], best_hsv[0], best_hsv[1], morph_radius=int(best_hsv[2])))

print('=' * 60)
print(f'{"Method":<42}  {"Test Dice":>10}')
print('-' * 60)
print(f'{"Lab 01 — hand-picked grayscale":<42}  {d_lab01:10.4f}')
print(f'{"Lab 02 — Bayesian opt (grayscale)":<42}  {d_lab02:10.4f}')
print(f'{"Lab 03-v2 Option A — B−R diff + morphology":<42}  {d_rgb:10.4f}')
print(f'{"Lab 03-v2 Option B — HSV + morphology":<42}  {d_hsv:10.4f}')
print('=' * 60)

## 8. N/C Ratio Prediction & Pearson Correlation Coefficient

A high Dice score tells us the *shape* of the predicted mask is close to ground truth.
But the clinical question is different: how accurately does our model estimate the
*nucleus-to-cytoplasm (N/C) ratio*?

$$\text{N/C ratio} = \frac{\text{nucleus pixels}}{\text{cytoplasm pixels}}$$

A high N/C ratio is a hallmark of malignancy.  We evaluate two things:

1. **Scatter plot** — predicted vs ground-truth ratio.  Points on the diagonal = perfect prediction.
2. **Pearson correlation coefficient** *r* — how linearly correlated are the predictions?
   A value of *r* = 1.0 is a perfect linear relationship; *r* = 0 means no correlation.

$$r = \frac{\sum_i (p_i - \bar{p})(g_i - \bar{g})}{\sqrt{\sum_i (p_i - \bar{p})^2 \sum_i (g_i - \bar{g})^2}}$$

`np.corrcoef` computes this directly — no extra imports needed.

In [ ]:
def nc_ratio(mask):
    """N/C ratio from a (H, W) integer mask.  Returns NaN if cytoplasm is absent."""
    nuc  = (mask == 2).sum()
    cyto = (mask == 1).sum()
    return float('nan') if cyto == 0 else nuc / cyto

# Ground-truth N/C ratios for the test set
gt_ratios = np.array([nc_ratio(y[i]) for i in test_idx])
valid     = ~np.isnan(gt_ratios)
gt        = gt_ratios[valid]
idx_v     = test_idx[valid]

print(f'Valid test images: {valid.sum()} / {len(test_idx)}')
print(f'Ground-truth N/C — mean: {gt.mean():.3f}  std: {gt.std():.3f}  range: [{gt.min():.3f}, {gt.max():.3f}]')

# Predicted N/C ratios
pred_lab01 = np.array([nc_ratio(segment_gray(X[i], 0.45, 0.85))                                                     for i in idx_v])
pred_lab02 = np.array([nc_ratio(segment_gray(X[i], best_gray[0], best_gray[1]))                                     for i in idx_v])
pred_rgb   = np.array([nc_ratio(segment_rgb( X[i], best_rgb[0], best_rgb[1], morph_radius=int(best_rgb[2])))        for i in idx_v])
pred_hsv   = np.array([nc_ratio(segment_hsv( X[i], best_hsv[0], best_hsv[1], morph_radius=int(best_hsv[2])))       for i in idx_v])

def pearson_r(pred, gt_vals):
    ok = ~np.isnan(pred)
    return np.corrcoef(pred[ok], gt_vals[ok])[0, 1]

print()
print(f'Pearson r:')
print(f'  Lab 01 (hand-picked):  {pearson_r(pred_lab01, gt):.4f}')
print(f'  Lab 02 (Bayes gray):   {pearson_r(pred_lab02, gt):.4f}')
print(f'  Option A (B−R diff):   {pearson_r(pred_rgb,   gt):.4f}')
print(f'  Option B (HSV):        {pearson_r(pred_hsv,   gt):.4f}')

In [ ]:
lim = max(gt.max(),
          np.nanmax(pred_lab01), np.nanmax(pred_lab02),
          np.nanmax(pred_rgb),   np.nanmax(pred_hsv)) * 1.1

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (title, pred, color) in zip(axes, [
    ('Lab 01 — hand-picked',   pred_lab01, 'steelblue'),
    ('Lab 02 — Bayes gray',    pred_lab02, 'darkorange'),
    ('Option A — B−R diff',    pred_rgb,   'seagreen'),
    ('Option B — HSV',         pred_hsv,   'mediumpurple'),
]):
    ok = ~np.isnan(pred)
    r  = pearson_r(pred, gt)
    ax.scatter(gt[ok], pred[ok], alpha=0.6, color=color, s=30)
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1)
    ax.set_xlabel('Ground-truth N/C ratio')
    ax.set_ylabel('Predicted N/C ratio')
    ax.set_title(f'{title}\nr = {r:.4f}')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print('Points near the dashed diagonal = accurate N/C ratio predictions.')

## Wrap-up

**Key takeaways:**

- **Colour is information; grayscale discards it.**  Converting H&E images to grayscale collapses
  purple nuclei and pink cytoplasm onto the same brightness axis.  Both B−R and HSV recover
  the colour distinction with only a few lines of arithmetic.

- **B−R difference (Option A)** is the simplest possible colour feature: one subtraction.
  It requires a grayscale brightness threshold for background, but nucleus detection is
  directly physics-driven (purple absorbs red, transmits blue).

- **HSV (Option B)** separates the three classes with orthogonal signals: saturation for
  background, hue for nucleus vs cytoplasm.  No brightness threshold is needed for background
  because unsaturated = white regardless of absolute brightness.

- **Which is better? Look at your numbers.**  If one approach has noticeably higher Dice or
  Pearson *r*, inspect the feature maps (§2) to understand why.  The winning representation
  will be the one whose feature values for the three classes overlap least.

- **The grayscale ceiling is a representation problem, not a threshold problem.**  No amount
  of morphological polish can recover information that was thrown away at the representation stage.

In the next lab we abandon fixed thresholds entirely and treat each pixel as a point in
3-D colour space, training a k-NN classifier to separate the classes directly.

---

## Group Exercise — Comparing Representations

Each person evaluates one comparison on `test_idx[0:5]`.

| Person | Task |
|---|---|
| A | Print mean B−R per class for your image.  Does the ordering (nucleus > background > cytoplasm) hold? |
| B | Print mean S and H per class for your image.  Does S cleanly separate background from stained pixels? |
| C | Run both `segment_rgb` and `segment_hsv` with `use_morph=False`.  How much does morphology contribute when colour already provides better class separation? |

Share your numbers and discuss: which representation — B−R or HSV — is more informative for
this dataset, and why?